In [ ]:
!pip install -q pandas requests

In [ ]:
import requests
import json
import pandas as pd

# The direct URL to the JSON file on your MinIO console
DATA_URL = 'http://127.0.0.1:9001/browser/spotify-raw-data/mpd.slice.114000-114999.json'

try:
    # 1. Fetch the data from the URL
    response = requests.get(DATA_URL)
    response.raise_for_status() # Check for HTTP errors (4xx or 5xx)

    # 2. Parse the JSON content
    data = response.json()
    
    print("Data successfully fetched and parsed!")
    
except requests.exceptions.RequestException as e:
    print(f"Error accessing the MinIO server or file: {e}")
    print("Ensure MinIO is running and accessible at http://127.0.0.1:9001.")
except json.JSONDecodeError as e:
    print(f"Error decoding JSON: {e}")
    print("The file might not be valid JSON.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
!pip install minio


In [ ]:
from minio import Minio
import json

# Kh?i t?o client
client = Minio("127.0.0.1:9000", access_key="minioadmin", secret_key="minioadmin", secure=False)

# T?i file JSON t? bucket
for b in client.list_buckets():
    print(b.name)
data = client.get_object("spotify-raw-data", "mpd.slice.114000-114999.json")

# ??c n?i dung JSON
playlist_data = json.load(data)
print(len(playlist_data["playlists"]))


In [ ]:
playlists = playlist_data['playlists']
records = []

for p in playlists:
    for t in p['tracks']:
        records.append({
            'playlist_id': p['pid'],
            'playlist_name': p.get('name', ''),
            'track_name': t.get('track_name', ''),
            'artist_name': t.get('artist_name', ''),
            'album_name': t.get('album_name', '')
        })

df = pd.DataFrame(records)

print("? S? l??ng playlist:", df['playlist_id'].nunique())
print("? T?ng s? track entries:", len(df))
print("? S? l??ng track duy nh?t:", df['track_name'].nunique())
print(df.head())

In [ ]:
avg_tracks = df.groupby('playlist_id')['track_name'].count().mean()
print(f"S? b?i h?t trung b?nh m?i playlist: {avg_tracks:.2f}")

# Ngh? s? ph? bi?n nh?t
top_artists = df['artist_name'].value_counts().head(10)
print("\nTop 10 ngh? s? ph? bi?n nh?t:")
print(top_artists)

# Album ph? bi?n nh?t
top_albums = df['album_name'].value_counts().head(10)
print("\nTop 10 album ph? bi?n nh?t:")
print(top_albums)

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Spotify Production Pipeline") \
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "600000")  \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "600000")\
    .config("spark.hadoop.fs.s3a.attempts.maximum", "100")\
    .config("spark.hadoop.fs.s3a.retry.limit", "20")\
    .config("spark.sql.caseSensitive", "true") \
    .getOrCreate()


print("Spark Session Ready!")

In [ ]:
# 1. ??nh ngh?a Schema chu?n c?a Spotify MPD
# Vi?c n?y gi?p Spark ??c nhanh h?n v? lo?i b? b?n ghi sai format ngay t? ??u
track_schema = StructType([
    StructField("pos", IntegerType(), True),
    StructField("artist_name", StringType(), True),
    StructField("track_uri", StringType(), True),
    StructField("artist_uri", StringType(), True),
    StructField("track_name", StringType(), True),
    StructField("album_uri", StringType(), True),
    StructField("duration_ms", LongType(), True),
    StructField("album_name", StringType(), True)
])

playlist_schema = StructType([
    StructField("name", StringType(), True),
    StructField("collaborative", StringType(), True),
    StructField("pid", LongType(), True),
    StructField("modified_at", LongType(), True),
    StructField("num_tracks", IntegerType(), True),
    StructField("num_albums", IntegerType(), True),
    StructField("num_followers", IntegerType(), True),
    StructField("tracks", ArrayType(track_schema), True), # Nested Array
    StructField("description", StringType(), True)
])
info_schema = StructType([
    StructField("tag", StringType(), True),
    StructField("generated_on", LongType(), True),
    StructField("slice", StringType(), True),
    StructField("version", StringType(), True)
])

root_schema = StructType([
    StructField("info", info_schema, True),
    StructField("playlists", ArrayType(playlist_schema), True)
])

print("Loading Raw Data from Data Lake...")
# ??c d? li?u v?i schema ?? ??nh ngh?a
raw_df = spark.read \
    .schema(root_schema) \
    .option("mode", "PERMISSIVE") \
    .option("multiline", "true") \
    .json("s3a://spotify-raw-data/mpd.slice.114000-114999.json") # Demo 1 file

print(f"Raw Data Loaded.")

In [ ]:
print("Processing Silver Layer...")

# 1. Explode Playlists (T?ch m?ng playlist th?nh t?ng d?ng)
playlists_df = raw_df.select(explode(col("playlists")).alias("p"))

# 2. Explode Tracks (T?ch m?ng tracks l?ng b?n trong playlist)
# ??y l? b??c "Explode nested track arrays" trong y?u c?u
flat_df = playlists_df.select(
    col("p.pid").alias("playlist_id"),
    col("p.name").alias("playlist_name"),
    col("p.num_tracks"),
    col("p.num_followers"),
    col("p.modified_at"),
    explode(col("p.tracks")).alias("t")
)

# 3. Parse & Normalize Metadata (L?m s?ch)
# - Trim: C?t kho?ng tr?ng th?a
# - Lower: Chuy?n v? ch? th??ng ?? chu?n h?a
# - Cast: ??m b?o ??ng ki?u d? li?u
silver_df = flat_df.select(
    col("playlist_id"),
    trim(col("playlist_name")).alias("playlist_name"),
    col("num_tracks"),
    col("num_followers"),
    # Convert timestamp (modified_at l? epoch seconds)
    to_timestamp(col("modified_at")).alias("modified_date"),
    
    # Track Metadata Parsing
    trim(col("t.track_uri")).alias("track_uri"),
    trim(col("t.track_name")).alias("track_name"),
    trim(col("t.artist_uri")).alias("artist_uri"),
    trim(col("t.artist_name")).alias("artist_name"),
    col("t.duration_ms"),
    
    # Th?m c?t partition key (V? d?: l?y ch? c?i ??u c?a artist ?? partition cho ??u)
    substring(col("t.artist_name"), 0, 1).alias("artist_partition")
).drop_duplicates() # Lo?i b? d?ng tr?ng l?p (Deduplication)

# L?u b?ng s?ch n?y xu?ng Data Lake (Silver)
silver_df.write.mode("overwrite").parquet("s3a://warehouse/silver/tracks")
print("Silver Layer (Cleaned & Normalized) Saved.")
silver_df.show(3)

In [ ]:
print("Processing Gold Layer (Feature Engineering)...")

# Feature 1: Artist Frequency (?? ph? bi?n c?a ngh? s?)
# Ngh? s? n?y xu?t hi?n trong bao nhi?u playlist? T?ng th?i l??ng nghe l? bao nhi?u?
artist_features = silver_df.groupBy("artist_name") \
    .agg(
        countDistinct("playlist_id").alias("total_playlist_appearances"),
        count("track_uri").alias("total_tracks_count"),
        avg("duration_ms").alias("avg_track_duration"),
        sum("duration_ms").alias("total_listen_time_ms")
    ) \
    .withColumn("popularity_score", col("total_playlist_appearances") * 0.7 + col("total_tracks_count") * 0.3)

# Feature 2: Playlist Complexity (?? ph?c t?p c?a Playlist)
# Playlist n?y c? bao nhi?u ngh? s? kh?c nhau? ?? ?a d?ng th? n?o?
playlist_features = silver_df.groupBy("playlist_id", "playlist_name") \
    .agg(
        count("track_uri").alias("playlist_length"),
        countDistinct("artist_name").alias("unique_artists"),
        avg("duration_ms").alias("avg_track_duration")
    ) \
    .withColumn("diversity_ratio", col("unique_artists") / col("playlist_length"))

print("Sample Artist Features:")
artist_features.orderBy(desc("total_playlist_appearances")).show(5)

print("Sample Playlist Features:")
playlist_features.orderBy(desc("diversity_ratio")).show(5)

In [ ]:
print("Saving Final Data (Storage Optimization)...")

# 1. L?u Artist Features (Partition theo ch? c?i ??u ?? truy v?n nhanh)
# L?u ?: Partition by artist_id/name tr?c ti?p s? t?o ra h?ng tri?u folder nh? (Bad Practice).
# Chu?n ch? l? n?n partition theo nh?m (Bucketing) ho?c ch? c?i ??u.
artist_features.write \
    .mode("overwrite") \
    .partitionBy("total_playlist_appearances") \
    .parquet("s3a://warehouse/gold/artist_features")

# 2. L?u Playlist Features (Partition by Range v? d? theo ?? d?i playlist)
# T?o bucket c?t playlist_length ?? partition cho g?n
playlist_features_bucketed = playlist_features.withColumn(
    "length_bucket", 
    (col("playlist_length") / 50).cast("integer") * 50 # Gom nh?m 0-50, 50-100...
)

playlist_features_bucketed.write \
    .mode("overwrite") \
    .partitionBy("length_bucket") \
    .parquet("s3a://warehouse/gold/playlist_features")

print("All Data Saved & Partitioned Successfully!")